In [1]:
import os
import re
import psutil
from tqdm import tqdm

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

import warnings
warnings.filterwarnings('ignore')

pool = ThreadPoolExecutor()
default_workers_threads = pool._max_workers

print(f"CPU count: {os.cpu_count()}")
print(f"Memory GB: {psutil.virtual_memory().total >> 30}")
print(f"Default thread workers: {default_workers_threads}")

CPU count: 12
Memory GB: 15
Default thread workers: 16


In [2]:
num_workers = 8

# Arxiv нэг хуудсанд гарах өгүүллэлийг тоог 200 гэж заасан тул хуудас хооронд 200-р шилжинэ (increment)
start_page = 0
end_page = 4000
increment = 200

# Advanced search дээр хайлтын түлхүүр үгээ оруулна. Энэ тохиолдолд 'audio recognition'
search_term = 'audio+recognition'
base_link = f"https://arxiv.org/search/advanced?advanced=&terms-0-operator=AND&terms-0-term={search_term}&terms-0-field=all&classification-physics_archives=all&classification-include_cross_list=include&date-filter_by=all_dates&date-year=&date-from_date=&date-to_date=&date-date_type=submitted_date&abstracts=show&size=200&order=-announced_date_first"

papers = []

# Хуудас бүрийн мэдээллийг авахад бид олон удаа request үүсгэж байгаа. Иймд нэг удаа үүсгэсэн connection-г олон дахин ашиглах үүднээс 
# session тодорхойлно
def scrape_papers(start_num):
    session = requests.Session()

    linkie = f"{base_link}&start={start_num}"

    html_text = session.get(linkie).text
    soup = BeautifulSoup(html_text, 'html.parser')

    papers_sub = soup.find_all('li', class_ = 'arxiv-result')

    papers.extend(papers_sub)

    session.close()

In [3]:
# 2-р хэсэгт зохиолч бүрийн мэдээллийг нэг нэгээр хуулах нь дор хаяж 1 цаг болж байсан. 
# Иймд эхлээд 2-р хэсэгт параллель тооцоо ашигласан ба энэ хэсэгт ердөө
# хийсэн зүйлээ бататгах зорилгоор ашигласан юм.
with ThreadPoolExecutor(max_workers = num_workers) as executor, tqdm(total = (end_page - start_page), 
                                                                     desc = f'Scraping papers from arxiv, [SEARCH TERM]: {search_term}') as pbar:
    futures = [executor.submit(scrape_papers, start_num) for start_num in range(start_page, end_page, increment)]
    for future in concurrent.futures.as_completed(futures):
        pbar.update(increment)

print(f"Number of papers scraped: {len(papers)}")

Scraping papers from arxiv, [SEARCH TERM]: audio+recognition: 100%|██████████| 4000/4000 [00:19<00:00, 208.53it/s]

Number of papers scraped: 4000


In [4]:
def punct(full_text):
    temp = [sent.strip() for sent in re.findall("""\s+[^.!?]*[.!?]""", full_text)]
    temp = re.sub('\..', '.', '. '.join(temp))
    temp = re.sub('\s+[a-zA-Z]\.', '', temp)
    
    return temp

def get_subjects(paper_arxiv_code, arxiv_base_link = "https://arxiv.org/abs/"):
    session = requests.Session()

    link = arxiv_base_link + paper_arxiv_code
    response = requests.get(link)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        subject_tag = soup.find('td', class_='tablecell subjects')
        if subject_tag:
            paper_subjects = subject_tag.text.strip()
        else:
            paper_subjects = np.nan
    else:
        paper_subjects = np.nan

    session.close()

    return paper_subjects

patterns = {
    'submitted to': r'submitted to (.+)',
    'Accepted to': r'Accepted to (.+)',
    'Accepted in': r'Accepted in (.+)',
    'accepted by': r'accepted by (.+)',
    'Journal ref': r'Journal ref: (.+)'
}

In [5]:
paper = papers[np.random.randint(0, len(papers))]

paper_arxiv_code = paper.find('p', class_ = 'list-title is-inline-block').text.split('\n')[0]

paper_subjects = get_subjects(paper_arxiv_code)

# Paper Title
paper_title = paper.find('p', class_ = 'title is-5 mathjax').text.strip()

# Authors
paper_authors = paper.find('p', class_ = 'authors')
paper_authors = ','.join([name.text.strip() for name in paper_authors.find_all('a')])

# Abstract
paper_abstract = paper.find('p', class_ = 'abstract mathjax')
paper_abstract = punct(paper_abstract.find('span', class_ = 'abstract-full has-text-grey-dark mathjax').text)

# Comment, which includes submitted Journals etc.
# Аливаа өгүүллэгийн хувьд хэвлэгдсэн сэтгүүлийн мэдээлэл нь цөөн тооны pattern-ийн дагуу бичигдсэн байсан.
# Иймд энэ хэсэгт pattern бүрийг шалгана.
paper_comment = paper.find('p', class_='comments is-size-7')
paper_journal = np.nan
if paper_comment:
    paper_comment_text = re.sub(r'\s+', ' ', paper_comment.text.strip())
    for keyword, pattern in patterns.items():
        if keyword in paper_comment_text:
            match = re.search(pattern, paper_comment_text)
            if match:
                paper_journal = match.group(1).strip()
                break
# Submission date
# Сүүлд нь бүх хугацааг харсан. Тэгэхэд хугацаа бүр ижил форматтай байсан.
# Иймд ганц удаа split ашиглаж илгээсэн огноог гаргах боломжтой байсныг ойлгосон. Гэхдээ анх ялгаатай pattern-тай огноо байх вий хэмээн болгоомжилж
# бичсэн аргаа үлдээсэн.
submit_date = paper.find('p', class_ = 'is-size-7').text.split(';')[0]
submit_date = re.findall('([0-9]{1,2}\s(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|(Nov|Dec)(?:ember)?)\, [0-9]{4})', submit_date)
submit_date = [sent for sent in submit_date[0] if len(sent)][0]

originally_announced_date = paper.find('p', class_ = 'is-size-7').text.strip().split('originally announced ')[-1][:-1]

In [6]:
df = pd.DataFrame({'title': [paper_title], 
                   'authors': [paper_authors],
                   'abstract': [paper_abstract],
                   'Journal': [paper_journal],
                   'code': [paper_arxiv_code],
                   'subjects': [paper_subjects],
                   'submitted_date': [submit_date],
                   'orginally_announced_date': [originally_announced_date]})
df['submitted_date'] = pd.to_datetime(df['submitted_date'])
df['orginally_announced_date'] = pd.to_datetime(df['orginally_announced_date'], errors = 'coerce')

df

,title,authors,abstract,Journal,code,subjects,submitted_date,orginally_announced_date
0,Zorro: the masked multimodal transformer,"Adrià Recasens,Jason Lin,Joāo Carreira,Drew Ja...",Attention-based models are appealing for multi...,NaN,arXiv:2301.09595,Computer Vision and Pattern Recognition (cs.CV),2023-02-22,2023-01-01


In [8]:
df['subjects']

0    Computer Vision and Pattern Recognition (cs.CV)
Name: subjects, dtype: object

In [17]:
subjects = "Computer Vision and Pattern Recognition (cs.CV); Machine Learning (cs.LG); Multimedia (cs.MM); Audio and Speech Processing (eess.AS)"

# Regular expression pattern to match strings inside parentheses and strings before them
pattern = r'([^()]+) \(([^()]+)\)'
pattern = r'([^();]+) \(([\w.]+)\)'

matches = re.findall(pattern, subjects)

# Extracted strings
before_list = []
inside_list = []

# Iterate over matches and populate the lists
for before, inside in matches:
    before_list.append(before.strip())
    inside_list.append(inside.strip())

print("Strings before parentheses:", before_list)
print("Strings inside parentheses:", inside_list)

Strings before parentheses: ['Computer Vision and Pattern Recognition', 'Machine Learning', 'Multimedia', 'Audio and Speech Processing']
Strings inside parentheses: ['cs.CV', 'cs.LG', 'cs.MM', 'eess.AS']
